In [2]:
from langchain_openai import ChatOpenAI # pip install langchain-openai
model = ChatOpenAI(model="gpt-4o-mini")

from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 귀여운 여인에 나오는 Vivian Ward야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 Edward Lewis입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요??"),
]

model.invoke(messages)

AIMessage(content='안녕, Edward! 물론이지, 저녁 같이 먹는 거 좋지! 어떤 곳에 가고 싶어? 당신과 함께라면 어디든 좋을 것 같아.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 61, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_73c5e31c8b', 'id': 'chatcmpl-EdUv8zezw8TKXyShxkvhIkW250Z7g', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17c09-78d0-72a9-a0b8-2643c53453bc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 38, 'total_tokens': 99, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'a

In [3]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()       # Create a parser instance

result = model.invoke(messages)  # Get the model's output
parser.invoke(result)            # Parse the output to a string  

'안녕하세요, 에드워드! 저녁 초대 정말 고맙습니다. 당신과 함께하는 저녁, 항상 멋진 경험이 될 것 같아요! 어디로 갈까요?'

In [4]:
chain = model | parser
chain.invoke(messages)

'안녕하세요, Edward! 물론이죠, 저녁 같이 먹는 건 정말 좋을 것 같아요! 어디서 만나면 좋을까요? 어떤 요리를 좋아하시나요? 기대돼요! ✨'

In [5]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_1} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_2}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "귀여운 여인",
    "character_1": "비비안",
    "character_2": "에드워드",
    "activity": "저녁"
})

print(result)

messages=[SystemMessage(content='너는 귀여운 여인에 나오는 비비안 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 에드워드입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [7]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "Pretty Woman",
    "character_1": "비비안",
    "character_2": "에드워드",
    "activity": "저녁"
})

'안녕, 에드워드. 저녁 약속이라니, 뭔가 특별한 날이 될 것 같네요. 어떤 데이트를 계획하고 있나요? 제가 좋게 대할 수 있도록 좀 더 알려줘요! '

In [8]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "귀여운 여인",
    "character_1": "비비안",
    "character_2": "에드워드",
    "activity": "저녁"
})

Adlib(answer='안녕하세요, 에드워드! 당신과 저녁을 함께 하는 건 정말 좋겠어요. 저녁 메뉴는 어떤 게 좋을까요? 혹시 제가 좋아하는 스테이크를 먹으러 가는 건 어때요? 그리고, 오늘 기분도 어떤지 궁금해요!', main_emotion='기쁨', main_emotion_intensity=0.8)